# Exercise 2.1: Loading, Inspecting and Cleaning (Angola IEA)

This notebook uses `IEA_2025_IV_TRIM_IND.sav`, the individual file of the
Inquerito ao Emprego em Angola (IEA), 4th quarter 2025, published by INE Angola.

It is a real SPSS export: 53,353 people, 206 columns, all variable and value
labels in Portuguese.

You will practice:
- Loading an SPSS file with `pd.read_spss()` and keeping its value labels
- Running structural diagnostics with `describe()` and `value_counts()`
- Telling a DataFrame from a Series, and reading dtypes critically
- Recasting columns the labels got wrong, and identifiers the file got wrong
- Turning a YYYYMMDD number into a real date
- Telling missing by design apart from missing by error
- Finding duplicates on a compound key
- Checking a rule that `describe()` cannot see

> **Pipeline:** reads `0_raw/`, writes a cleaned file to `10_cleaned/`.

### Path Setup (run first)

> Use `os.path.join` for path construction. Define the country folder once, then
> join the sub folder and the file name onto it.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_RAW_DIR = '../../data/0_raw/angola'
DATA_CLEAN_DIR = '../../data/10_cleaned'

EMPLOYMENT_DIR = 'employment_survey'
RAW_FILE = 'IEA_2025_IV_TRIM_IND.sav'
raw_path = os.path.join(DATA_RAW_DIR, EMPLOYMENT_DIR, RAW_FILE)

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Data path:', raw_path)
print('Exists?:', os.path.exists(raw_path))

---

## Task 1: Load the file and take a first look

`pd.read_spss()` reads SPSS `.sav` files. Keep `convert_categoricals=True`, the
default, so coded variables arrive as the readable Portuguese labels stored in
the file rather than as bare numbers.

206 columns is more than any single analysis needs, so `usecols` keeps the 27
that carry the demographic core, the labour module, the interview date and the
survey weight. Two columns that look useful, `G_12` and `G_13`, are deliberately
left out: they are empty in every row of this extract, so there is no reason to
load them at all.

The column names come from the questionnaire and are shouted in upper case.
Lower casing them is enough to make them ordinary snake case; nothing is renamed,
so every name still matches the official codebook.

In [ ]:
SPSS_COLS = [
    'NIDF', 'PPNO', 'G_06_ID_IEA', 'PROV', 'AREA_RESID', 'G_15_TRIMESTRE',
    'DEM_REL', 'DEM_SEX', 'DEM_AGE', 'DEM_MRT', 'DEM_EDL', 'S03_01',
    'ATW_PAY', 'ATW_PFT', 'ATW_FAM', 'ABS_JOB',
    'SRH_JOB', 'SRH_BUS', 'SRH_AVN', 'SRH_AVL', 'SRH_DES',
    'WKT_USHRSTOT', 'WKT_ACHRSTOT', 'MJT_SYR', 'MJJ_EMP_REL', 'GHVEDT',
    'POND_IEA_IV_TRIM_2025_IND',
]

df = pd.read_spss(  # your code here: raw_path, usecols=SPSS_COLS, convert_categoricals=True )
df.columns =   # your code here: lower case the column names

print('Loaded:', df.shape)
df.head()

In [ ]:
df.  # your code here: last 5 rows

**Questions:**

- How many rows and columns did you load? What is one row?
- Look at `prov` and `dem_sex`. Are they text or numbers? What made them so?
- The names are the questionnaire's own rather than friendly English. What do you
  gain by keeping them, and what do you lose?

---

## Task 2: Summary statistics with `describe()`

`describe(include='all')` covers text and numbers in one table. Look hard at
every `min` and `max`, and at any `count` below 53,353.

In [ ]:
df.describe(  # your code here: include='all' ).T

**Questions:**

- Look at the `max` of `wkt_ushrstot`. Is that a possible working week? What is it?
- `dem_age` runs 0 to 120. Which end is a real value and which is not?
- What is `ghvedt` really, and why is its mean meaningless?
- Read the `count` row. Which columns are answered by only a fifth of the sample,
  and can you guess why?

---

## Task 3: Explore categories with `value_counts()`

`value_counts()` is the fastest way to see what is actually in a coded column.
Always pass `dropna=False` so the gaps are counted too.

In [ ]:
print(df['prov'].  # your code here: value_counts(dropna=False).sort_index() )

In [ ]:
print(df['area_resid'].value_counts(dropna=False))
print()
print(df['dem_sex'].value_counts(dropna=False))
print()
print(df['dem_edl'].value_counts(dropna=False).sort_index())

In [ ]:
print('Distinct households:', df['nidf'].nunique())
print('Rows per household, describe:')
print(df['nidf'].value_counts().describe())

**Questions:**

- How many provinces appear, and which is largest?
- `dem_edl` has a category meaning "no level at all". Why would sorting this
  column as if it were ordinal be wrong?
- How many households are there, and how many people per household on average?

---

## Task 4: DataFrame vs Series, and reading the dtypes

Selecting one column returns a **Series**, a one dimensional object with its own
dtype and its own methods. `.str`, `.dt` and `.value_counts()` all belong to
Series, not to the whole DataFrame.

The dtypes are where the next hour of work is decided, so read them properly.

In [ ]:
print(type(df))
print(type(df['dem_age']))

In [ ]:
df.  # your code here

**Questions:**

- Which columns are `category` and which are `float64`? What decided that?
- Three dtypes are wrong for what the column actually means. Find them.
- `mjt_syr` is a year. What dtype did it get, and can you subtract years from it?

---

## Task 5: Recast the columns the file got wrong

Applying value labels is convenient, but it is applied to every labelled variable,
including ones where the label marks a sentinel rather than a category.

`mjt_syr` is the year somebody started their main job. The value `9997` carries
the label `NÃO SABE`, so pandas concludes the whole column is categorical and a
year you cannot subtract is useless. Casting back with `pd.to_numeric` and
`errors='coerce'` fixes it in one move: every real year converts, and the text
label becomes `NaN`, which is exactly what "does not know" means.

The hours columns have the same `997` sentinel but no label for it, so they stay
numeric and the sentinel has to be replaced by hand.

In [ ]:
print('mjt_syr dtype before:', df['mjt_syr'].dtype)
print('categories include:', [c for c in df['mjt_syr'].cat.categories if not isinstance(c, float)])

df['mjt_syr'] = pd.to_numeric(  # your code here: cast to object, errors='coerce' )

print()
print('mjt_syr dtype after: ', df['mjt_syr'].dtype)
print('range:', df['mjt_syr'].min(), 'to', df['mjt_syr'].max())
print('mean: ', round(df['mjt_syr'].mean(), 1))

In [ ]:
# 997 is the same "does not know" sentinel, but unlabelled, so it stayed numeric.
print('hours max before:', df['wkt_ushrstot'].max())

df['wkt_ushrstot'] = df['wkt_ushrstot'].  # your code here: replace the sentinels with np.nan
df['wkt_achrstot'] = df['wkt_achrstot'].  # your code here: replace the sentinels with np.nan

print('hours max after: ', df['wkt_ushrstot'].max())

In [ ]:
# Identifiers are labels, not quantities. Float to integer to string, or the
# trailing .0 survives and joins to nothing.
print('Before:', df['nidf'].head(3).tolist())

for col in ['nidf', 'ppno', 'g_06_id_iea']:
    df[col] =   # your code here: int64 then string

print('After: ', df['nidf'].head(3).tolist())

In [ ]:
# ghvedt is the float 20251204.0, meaning 2025-12-04. Int64 tolerates the missing
# values that plain int64 would reject.
date_text = df['ghvedt'].astype('Int64').astype('string')
df['ghvedt'] = pd.to_datetime(  # your code here: format='%Y%m%d', errors='raise' )

print('dtype:', df['ghvedt'].dtype)
print('Range:', df['ghvedt'].min(), 'to', df['ghvedt'].max())
print('Missing (NaT):', df['ghvedt'].isna().sum())
print()
print(df['ghvedt'].dt.month.value_counts(dropna=False).sort_index())

**Questions:**

- What dtype does `mjt_syr` end up with, and what is its mean once cast? What
  happened to the `NÃO SABE` entries?
- Why did the hours columns need a manual `replace` when `mjt_syr` did not?
- What does the identifier look like if you skip the `int64` step?
- Look at the date range and the month counts. Is this really the 4th quarter of
  2025? What is missing entirely?

---

## Task 6: What the `category` dtype is worth

`area_resid` holds two distinct values repeated 53,353 times. Stored as
`category` pandas keeps each label once and small integer codes alongside;
stored as plain text it keeps 53,353 separate strings.

In [ ]:
as_category = df['area_resid'].memory_usage(deep=True)
as_text =   # your code here: the same column as plain object, memory_usage(deep=True)

print(f'category: {as_category:,} bytes')
print(f'text:     {as_text:,} bytes')
print(f'Saved:    {(1 - as_category / as_text) * 100:.1f}%')

**Questions:**

- How much memory does the category form save over plain text on this column?
- Does the saving depend on the number of rows, or the number of categories?
- What happens to that saving when you write the file to CSV?

---

## Task 7: Subset to inspect the problems

Filtering here is for **looking**. Deciding what to remove comes later, and every
removal has to be justifiable.

In [ ]:
# Implausible working weeks, now that the sentinel is gone
print('wkt_ushrstot > 100:', (df['wkt_ushrstot'] > 100).sum())
df[df['wkt_ushrstot'] > 100][['nidf', 'wkt_ushrstot', 'wkt_achrstot']].head()

In [ ]:
# Combine conditions: each one needs its own parentheses
old_and_working = df[(df['dem_age'] >= 65) & (df['wkt_ushrstot'] > 40)]
print('People 65+ working over 40 hours:', len(old_and_working))
old_and_working[['nidf', 'dem_age', 'wkt_ushrstot']].head()

In [ ]:
# isin() for a set of provinces, and str.contains() for a text search
target = df[df['prov'].  # your code here: isin Luanda and Benguela ]
print('Rows in Luanda or Benguela:', len(target))

lunda = df[df['prov'].astype('string').str.contains('Lunda', na=False)]
print('Rows in the Lunda provinces:', len(lunda))
print(lunda['prov'].value_counts())

**Questions:**

- How many people report more than 100 usual hours a week? Are any of them
  sentinels at this point?
- Why must each condition be wrapped in parentheses when combining with `&`?
- What does `na=False` do in `str.contains()`, and what happens without it?

---

## Task 8: Detect missing values

Count them, express them as a share, and look at the shape of the problem before
deciding anything.

In [ ]:
missing = pd.DataFrame({
    'n_missing':   # your code here: count of missing per column
    'pct_missing':   # your code here: percent missing per column, rounded to 1
})
missing.sort_values('pct_missing', ascending=False)

In [ ]:
counts = df.isna().sum()
counts[counts > 0].sort_values().plot(kind='barh', color='coral', figsize=(9, 7))
plt.title('Missing values by column')
plt.xlabel('Count')
plt.tight_layout()
plt.show()

**Questions:**

- Which columns are most missing? Is that damage, or something else?
- What decides whether a gap is a problem?
- Look at the chart. Do you see one pattern or several?

---

## Task 9: Missing by design is not missing by error

The textbook first move is `dropna()`. On a survey with skip patterns it is a
catastrophe. Measure it before you trust it.

In [ ]:
print('Rows now:                   ', len(df))
print('Rows if we called dropna(): ', len(df.dropna()))

In [ ]:
# The right rule: only the identifiers are non negotiable
print('Before:', df.shape)
df = df.dropna(  # your code here: subset of the identifier columns )
print('After: ', df.shape)

**Questions:**

- How many rows survive `dropna()`? Were you expecting that?
- Which columns are genuinely non negotiable for a record to be usable?
- Would filling the labour columns with a median be reasonable here? Why not?

---

## Task 10: Duplicates on a compound key

No two rows here are identical, so `duplicated()` alone finds nothing. The real
key is the pair `nidf` plus `ppno`: one row per person per household.

In [ ]:
print('Exact duplicate rows:', df.duplicated().sum())

dup_mask = df.duplicated(  # your code here: subset of the compound key, keep=False )
print('Rows sharing a person key:', dup_mask.sum())
df[dup_mask].sort_values(['nidf', 'ppno'])[
    ['nidf', 'ppno', 'dem_age', 'dem_sex', 'dem_rel', 'wkt_ushrstot']]

In [ ]:
# Keep the most complete record in each group
df['missing_count'] = df.isna().sum(axis=1)

print('Before:', df.shape)
df = (
    df
    .sort_values(['nidf', 'ppno', 'missing_count'])
    .drop_duplicates(  # your code here: subset of the compound key, keep='first' )
    .drop(columns='missing_count')
)
print('After: ', df.shape)

**Questions:**

- How many exact duplicate rows are there? How many rows share a person key?
- Why does the compound key find what `duplicated()` alone cannot?
- Look at the ages within a duplicate group. Are these really the same person
  recorded twice? What does that mean for the rule you just applied?

---

## Task 11: A rule that `describe()` cannot catch

Every household should have exactly one head. No summary statistic will tell you
whether that holds, because it is a relationship between rows rather than a
property of a column.

In [ ]:
heads = df[df['dem_rel'] == 'Chefe/Pessoa de referência']
heads_per_household = heads['nidf'].  # your code here

print('Households with two or more heads:', (heads_per_household > 1).sum())
print('Households with no head recorded: ',
      df['nidf'].nunique() - heads_per_household.index.nunique())

**Questions:**

- How many households have two heads, and how many have none?
- Run this check before the deduplication as well. Does the number change? Why?
- Why can neither `describe()` nor `info()` catch this?

---

## Task 12: Save the cleaned dataset

Raw data is read only. Write the result to `10_cleaned/` and reload it to confirm
it survives the round trip.

In [ ]:
os.makedirs(DATA_CLEAN_DIR, exist_ok=True)
out_path = os.path.join(DATA_CLEAN_DIR, 'angola_iea_2025q4_clean.csv')

df = df.reset_index(drop=True)
df.  # your code here: to_csv with index=False
print('Saved:', out_path, '|', df.shape)

In [ ]:
check = pd.read_csv(out_path, dtype={'nidf': 'string', 'ppno': 'string',
                                     'g_06_id_iea': 'string'})
print('Reloaded:', check.shape)
print()
print(check[['nidf', 'prov', 'ghvedt', 'mjt_syr']].dtypes)
check[['nidf', 'prov', 'dem_age', 'ghvedt', 'mjt_syr']].head()

**Questions:**

- How many rows does the cleaned file have, and can you account for every row lost?
- Reload it and check the dtypes. Which ones did not survive, and why?
- What does `index=False` prevent?